In [1]:
# =========================================================
# STEP 1: Load SOURCE and TARGET data, then split TARGET
# New clean pipeline
# =========================================================

import os
import glob
import random
import numpy as np
import scipy.io as sio
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# ONLY CHANGE THESE TWO PATHS IF NEEDED
# -----------------------------
TARGET_BASE = r"G:\Data Sets\Cutting tool organized\Vibration_1440_1"
SOURCE_BASE = r"G:\Data Sets\Cutting tool organized\Vibration_1320_1"

# -----------------------------
# Settings
# -----------------------------
SOURCE_RPM = 1320
TARGET_RPM = 1440
TEST_SIZE = 0.20

CLASS_NAMES = ["BF", "GF", "N", "TF"]
CLASS_TO_ID = {"BF": 0, "GF": 1, "N": 2, "TF": 3}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

FOLDERS_TGT = {
    "BF": "Vib_BF1440_1",
    "GF": "Vib_GF1440_1",
    "N" : "Vib_N1440_1",
    "TF": "Vib_TF1440_1",
}

FOLDERS_SRC = {
    "BF": "Vib_BF1320_1",
    "GF": "Vib_GF1320_1",
    "N" : "Vib_N1320_1",
    "TF": "Vib_TF1320_1",
}

# -----------------------------
# Loader
# -----------------------------
def load_domain(base_path, folder_map, rpm_value, class_names):
    X, Y, D, FS, PATHS = [], [], [], [], []

    print(f"\nLoading domain: {rpm_value} rpm")
    for cls in class_names:
        folder = os.path.join(base_path, folder_map[cls])
        files = sorted(glob.glob(os.path.join(folder, "*.mat")))
        print(f"{rpm_value} rpm | {cls} | files = {len(files)}")

        if len(files) == 0:
            raise FileNotFoundError(f"No .mat files found in folder: {folder}")

        for fp in files:
            mat = sio.loadmat(fp)

            if "signals" not in mat:
                raise KeyError(f"'signals' not found in file: {fp}")
            if "fs" not in mat:
                raise KeyError(f"'fs' not found in file: {fp}")

            sig = np.array(mat["signals"], dtype=np.float32)
            fs_val = float(np.squeeze(mat["fs"]))

            if sig.ndim != 2:
                raise ValueError(f"'signals' must be 2D in file: {fp}, got shape {sig.shape}")
            if sig.shape[0] != 2:
                raise ValueError(f"'signals' first dimension must be 2 in file: {fp}, got shape {sig.shape}")
            if sig.shape[1] <= 0:
                raise ValueError(f"Invalid signal length in file: {fp}, got shape {sig.shape}")

            X.append(sig)
            Y.append(cls)
            D.append(rpm_value)
            FS.append(fs_val)
            PATHS.append(fp)

    return X, Y, D, FS, PATHS

# -----------------------------
# Load source and target
# -----------------------------
Xs, Ys, Ds, FSs, Ps = load_domain(SOURCE_BASE, FOLDERS_SRC, SOURCE_RPM, CLASS_NAMES)
Xt, Yt, Dt, FSt, Pt = load_domain(TARGET_BASE, FOLDERS_TGT, TARGET_RPM, CLASS_NAMES)

# -----------------------------
# Summary before split
# -----------------------------
print("\n================ LOADED SUMMARY ================")
print("Source samples:", len(Xs))
print("Target samples:", len(Xt))
print("Counts per class source:", dict(Counter(Ys)))
print("Counts per class target:", dict(Counter(Yt)))

# -----------------------------
# Stratified split on target
# target train -> unlabeled adaptation set
# target test  -> labeled evaluation set
# -----------------------------
y_target_ids = np.array([CLASS_TO_ID[y] for y in Yt], dtype=np.int64)

sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
idx_train_t, idx_test_t = next(sss.split(np.zeros(len(y_target_ids)), y_target_ids))

Xt_train = [Xt[i] for i in idx_train_t]
Yt_train = [Yt[i] for i in idx_train_t]
Dt_train = [Dt[i] for i in idx_train_t]
FSt_train = [FSt[i] for i in idx_train_t]
Pt_train = [Pt[i] for i in idx_train_t]

Xt_test = [Xt[i] for i in idx_test_t]
Yt_test = [Yt[i] for i in idx_test_t]
Dt_test = [Dt[i] for i in idx_test_t]
FSt_test = [FSt[i] for i in idx_test_t]
Pt_test = [Pt[i] for i in idx_test_t]

# -----------------------------
# Print split summary
# -----------------------------
print("\n================ TARGET SPLIT ================")
print("Target train (unlabeled) samples:", len(Xt_train))
print("Target test  (labeled) samples  :", len(Xt_test))
print("Counts per class target train:", dict(Counter(Yt_train)))
print("Counts per class target test :", dict(Counter(Yt_test)))

# -----------------------------
# Sanity checks
# -----------------------------
print("\n================ SANITY CHECKS ================")
print("Source example signal shape:", Xs[0].shape)
print("Target example signal shape:", Xt[0].shape)

print("Unique fs source:", sorted(list(set([round(v, 6) for v in FSs]))))
print("Unique fs target:", sorted(list(set([round(v, 6) for v in FSt]))))

assert len(Xs) > 0, "Source data is empty"
assert len(Xt) > 0, "Target data is empty"
assert len(Xt_train) > 0, "Target train split is empty"
assert len(Xt_test) > 0, "Target test split is empty"

assert Xs[0].shape[0] == 2, "Source signals must have 2 channels"
assert Xt[0].shape[0] == 2, "Target signals must have 2 channels"

print("\nStep 1 completed successfully.")


Loading domain: 1320 rpm
1320 rpm | BF | files = 112
1320 rpm | GF | files = 111
1320 rpm | N | files = 116
1320 rpm | TF | files = 112

Loading domain: 1440 rpm
1440 rpm | BF | files = 112
1440 rpm | GF | files = 110
1440 rpm | N | files = 112
1440 rpm | TF | files = 112

================ LOADED SUMMARY ================
Source samples: 451
Target samples: 446
Counts per class source: {'BF': 112, 'GF': 111, 'N': 116, 'TF': 112}
Counts per class target: {'BF': 112, 'GF': 110, 'N': 112, 'TF': 112}

================ TARGET SPLIT ================
Target train (unlabeled) samples: 356
Target test  (labeled) samples  : 90
Counts per class target train: {'GF': 88, 'N': 89, 'BF': 90, 'TF': 89}
Counts per class target test : {'GF': 22, 'N': 23, 'TF': 23, 'BF': 22}

================ SANITY CHECKS ================
Source example signal shape: (2, 25600)
Target example signal shape: (2, 25600)
Unique fs source: [25600.0]
Unique fs target: [25600.0]

Step 1 completed successfully.


In [2]:
# =========================================================
# STEP 2: Nonlinear feature extraction
# - Global nonlinear features: (N, 16)
# - Patch nonlinear features : (N, 100, 4)
# - Normalize global features using SOURCE only
# =========================================================

import math
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree

PATCH_SIZE = 256

# -----------------------------
# Helper functions
# -----------------------------
def _safe_std(x):
    s = float(np.std(x))
    return s if s > 1e-12 else 1e-12

def _embed(x, m, tau=1):
    x = np.asarray(x, dtype=np.float64)
    n = len(x) - (m - 1) * tau
    if n <= 1:
        return None
    return np.array([x[i:i + m * tau:tau] for i in range(n)], dtype=np.float64)

# -----------------------------
# Entropy and complexity features
# -----------------------------
def sampen_balltree(x, m=2, r=0.2, downsample=10):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    if len(x) < (m + 1) * 2 + 5:
        return 0.0

    sd = _safe_std(x)
    rad = r * sd

    Xm = _embed(x, m, 1)
    Xm1 = _embed(x, m + 1, 1)
    if Xm is None or Xm1 is None:
        return 0.0

    tree_m = BallTree(Xm, metric="chebyshev")
    tree_m1 = BallTree(Xm1, metric="chebyshev")

    Cm = tree_m.query_radius(Xm, r=rad, count_only=True).astype(np.float64) - 1.0
    Cm1 = tree_m1.query_radius(Xm1, r=rad, count_only=True).astype(np.float64) - 1.0

    Cm = np.maximum(Cm, 0.0)
    Cm1 = np.maximum(Cm1, 0.0)

    A = np.sum(Cm1) + 1e-12
    B = np.sum(Cm) + 1e-12
    return float(-np.log(A / B))

def apen_balltree(x, m=2, r=0.2, downsample=10):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    if len(x) < (m + 1) * 2 + 5:
        return 0.0

    sd = _safe_std(x)
    rad = r * sd

    Xm = _embed(x, m, 1)
    Xm1 = _embed(x, m + 1, 1)
    if Xm is None or Xm1 is None:
        return 0.0

    tree_m = BallTree(Xm, metric="chebyshev")
    tree_m1 = BallTree(Xm1, metric="chebyshev")

    Cm = tree_m.query_radius(Xm, r=rad, count_only=True).astype(np.float64)
    Cm1 = tree_m1.query_radius(Xm1, r=rad, count_only=True).astype(np.float64)

    Cm = Cm / (len(Xm) + 1e-12)
    Cm1 = Cm1 / (len(Xm1) + 1e-12)

    phi_m = np.mean(np.log(Cm + 1e-12))
    phi_m1 = np.mean(np.log(Cm1 + 1e-12))
    return float(phi_m - phi_m1)

def perm_entropy(x, order=5, delay=1, normalize=True, downsample=5):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    n = len(x)
    if n < order * delay + 2:
        return 0.0

    patterns = {}
    for i in range(n - delay * (order - 1)):
        window = x[i:i + delay * order:delay]
        key = tuple(np.argsort(window))
        patterns[key] = patterns.get(key, 0) + 1

    counts = np.array(list(patterns.values()), dtype=np.float64)
    p = counts / (np.sum(counts) + 1e-12)
    pe = -np.sum(p * np.log(p + 1e-12))

    if normalize:
        pe /= np.log(math.factorial(order) + 1e-12)
    return float(pe)

def weighted_perm_entropy(x, order=5, delay=1, normalize=True, downsample=5):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    n = len(x)
    if n < order * delay + 2:
        return 0.0

    weights = {}
    for i in range(n - delay * (order - 1)):
        window = x[i:i + delay * order:delay]
        key = tuple(np.argsort(window))
        w = float(np.var(window) + 1e-12)
        weights[key] = weights.get(key, 0.0) + w

    vals = np.array(list(weights.values()), dtype=np.float64)
    p = vals / (np.sum(vals) + 1e-12)
    wpe = -np.sum(p * np.log(p + 1e-12))

    if normalize:
        wpe /= np.log(math.factorial(order) + 1e-12)
    return float(wpe)

def higuchi_fd(x, kmax=8, downsample=2):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    n = len(x)
    if n < 20:
        return 1.0

    Lk = []
    ks = range(1, kmax + 1)

    for k in ks:
        Lm = []
        for m in range(k):
            idx = np.arange(m, n, k)
            if len(idx) < 2:
                continue
            diff = np.abs(np.diff(x[idx]))
            L = np.sum(diff) * (n - 1) / (len(idx) * k + 1e-12)
            Lm.append(L)
        if len(Lm) > 0:
            Lk.append(np.mean(Lm))

    if len(Lk) < 2:
        return 1.0

    lnL = np.log(np.array(Lk) + 1e-12)
    lnk = np.log(1.0 / (np.arange(1, len(Lk) + 1, dtype=np.float64) + 1e-12))
    slope = np.polyfit(lnk, lnL, 1)[0]
    return float(slope)

def katz_fd(x, downsample=2):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    n = len(x)
    if n < 5:
        return 1.0

    dists = np.abs(np.diff(x))
    L = np.sum(dists) + 1e-12
    d = np.max(np.abs(x - x[0])) + 1e-12
    return float(np.log10(n) / (np.log10(d / L) + np.log10(n) + 1e-12))

def lz_complexity(x, downsample=2):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    if len(x) < 20:
        return 0.0

    thr = np.median(x)
    s = (x > thr).astype(np.int8)

    i, k, l = 0, 1, 1
    c = 1
    n = len(s)

    while True:
        if i + k >= n or l + k >= n:
            break
        if s[i + k] == s[l + k]:
            k += 1
            if l + k >= n:
                c += 1
                break
        else:
            if k > 1:
                i += 1
                if i == l:
                    c += 1
                    l += k
                    if l >= n:
                        break
                    i, k = 0, 1
            else:
                i += 1
                if i == l:
                    c += 1
                    l += 1
                    if l >= n:
                        break
                    i, k = 0, 1

    return float(c / (n / np.log2(n + 1e-12) + 1e-12))

def hurst_rs(x, downsample=4):
    x = np.asarray(x, dtype=np.float64)[::downsample]
    n = len(x)
    if n < 50:
        return 0.5

    x = x - np.mean(x)
    y = np.cumsum(x)
    R = np.max(y) - np.min(y) + 1e-12
    S = _safe_std(x)
    return float(np.log(R / S) / (np.log(n + 1e-12) + 1e-12))

# -----------------------------
# Global nonlinear features
# 8 features per channel
# then channel-wise mean and std
# final: 16 dims
# -----------------------------
def nonlinear_vector_1ch(x):
    feats = []
    feats.append(sampen_balltree(x, m=2, r=0.2, downsample=10))
    feats.append(apen_balltree(x, m=2, r=0.2, downsample=10))
    feats.append(perm_entropy(x, order=5, delay=1, downsample=5))
    feats.append(weighted_perm_entropy(x, order=5, delay=1, downsample=5))
    feats.append(higuchi_fd(x, kmax=8, downsample=2))
    feats.append(katz_fd(x, downsample=2))
    feats.append(lz_complexity(x, downsample=2))
    feats.append(hurst_rs(x, downsample=4))
    return np.array(feats, dtype=np.float32)

def nonlinear_global_2ch(sig2ch):
    per_ch = []
    for ch in range(sig2ch.shape[0]):
        per_ch.append(nonlinear_vector_1ch(sig2ch[ch]))
    per_ch = np.stack(per_ch, axis=0)  # (2, 8)

    mean_feats = per_ch.mean(axis=0)   # (8,)
    std_feats  = per_ch.std(axis=0)    # (8,)
    g = np.concatenate([mean_feats, std_feats], axis=0)  # (16,)
    return g.astype(np.float32)

# -----------------------------
# Patch-level nonlinear features
# based on mean signal
# final per sample: (100, 4)
# -----------------------------
def sampen_small(x, m=2, r=0.2):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < (m + 1) * 2 + 2:
        return 0.0

    sd = _safe_std(x)
    rad = r * sd

    Xm = _embed(x, m, 1)
    Xm1 = _embed(x, m + 1, 1)
    if Xm is None or Xm1 is None:
        return 0.0

    dist_m = np.max(np.abs(Xm[:, None, :] - Xm[None, :, :]), axis=2)
    dist_m1 = np.max(np.abs(Xm1[:, None, :] - Xm1[None, :, :]), axis=2)

    np.fill_diagonal(dist_m, np.inf)
    np.fill_diagonal(dist_m1, np.inf)

    B = np.sum(dist_m <= rad) + 1e-12
    A = np.sum(dist_m1 <= rad) + 1e-12
    return float(-np.log(A / B))

def nonlinear_patch_mean(sig2ch, patch_size=PATCH_SIZE):
    x = sig2ch.mean(axis=0)
    L = x.shape[0]
    T = L // patch_size
    x = x[:T * patch_size].reshape(T, patch_size)

    patch_feats = []
    for t in range(T):
        xt = x[t]
        pe = perm_entropy(xt, order=4, delay=1, downsample=1)
        se = sampen_small(xt, m=2, r=0.2)
        hfd = higuchi_fd(xt, kmax=6, downsample=1)
        lzc = lz_complexity(xt, downsample=1)
        patch_feats.append([pe, se, hfd, lzc])

    return np.array(patch_feats, dtype=np.float32)  # (T, 4)

def extract_all_nonlinear(X_list):
    Pg = []
    Pp = []

    for i in range(len(X_list)):
        Pg.append(nonlinear_global_2ch(X_list[i]))
        Pp.append(nonlinear_patch_mean(X_list[i]))

        if (i + 1) % 100 == 0 or (i + 1) == len(X_list):
            print(f"Processed {i+1}/{len(X_list)} samples")

    Pg = np.vstack(Pg)
    Pp = np.stack(Pp, axis=0)
    return Pg, Pp

# -----------------------------
# Extract for source, target train, target test
# -----------------------------
print("\nExtracting source nonlinear features ...")
Pg_s, Pp_s = extract_all_nonlinear(Xs)

print("\nExtracting target train nonlinear features ...")
Pg_tu, Pp_tu = extract_all_nonlinear(Xt_train)

print("\nExtracting target test nonlinear features ...")
Pg_tt, Pp_tt = extract_all_nonlinear(Xt_test)

print("\n================ FEATURE SHAPES ================")
print("Global nonlinear shapes  :",
      "source =", Pg_s.shape,
      "| target_train =", Pg_tu.shape,
      "| target_test =", Pg_tt.shape)

print("Patch nonlinear shapes   :",
      "source =", Pp_s.shape,
      "| target_train =", Pp_tu.shape,
      "| target_test =", Pp_tt.shape)

# -----------------------------
# Normalize global nonlinear features
# using SOURCE only
# -----------------------------
scaler_global = StandardScaler()
Pg_s_norm = scaler_global.fit_transform(Pg_s)
Pg_tu_norm = scaler_global.transform(Pg_tu)
Pg_tt_norm = scaler_global.transform(Pg_tt)

print("\n================ NORMALIZATION CHECK ================")
print("Source normalized mean first 5:", np.round(Pg_s_norm.mean(axis=0)[:5], 4))
print("Source normalized std  first 5:", np.round(Pg_s_norm.std(axis=0)[:5], 4))

print("\n================ VALIDATION CHECKS ================")
print("Any NaN in source global :", np.isnan(Pg_s_norm).any())
print("Any NaN in target global :", np.isnan(Pg_tu_norm).any(), np.isnan(Pg_tt_norm).any())
print("Any NaN in source patch  :", np.isnan(Pp_s).any())
print("Any NaN in target patch  :", np.isnan(Pp_tu).any(), np.isnan(Pp_tt).any())

print("Patch length check source:", Pp_s.shape[1], "(expected 100)")
print("Patch length check target:", Pp_tu.shape[1], "(expected 100)")

print("\nExample source global feature first 6:", np.round(Pg_s[0][:6], 4))
print("Example source patch feature first patch:", np.round(Pp_s[0, 0], 4))

print("\nStep 2 completed successfully.")


Extracting source nonlinear features ...
Processed 100/451 samples
Processed 200/451 samples
Processed 300/451 samples
Processed 400/451 samples
Processed 451/451 samples

Extracting target train nonlinear features ...
Processed 100/356 samples
Processed 200/356 samples
Processed 300/356 samples
Processed 356/356 samples

Extracting target test nonlinear features ...
Processed 90/90 samples

================ FEATURE SHAPES ================
Global nonlinear shapes  : source = (451, 16) | target_train = (356, 16) | target_test = (90, 16)
Patch nonlinear shapes   : source = (451, 100, 4) | target_train = (356, 100, 4) | target_test = (90, 100, 4)

================ NORMALIZATION CHECK ================
Source normalized mean first 5: [-0. -0. -0.  0.  0.]
Source normalized std  first 5: [1. 1. 1. 1. 1.]

================ VALIDATION CHECKS ================
Any NaN in source global : False
Any NaN in target global : False False
Any NaN in source patch  : False
Any NaN in target patch  : Fals

In [3]:
# =========================================================
# STEP 3: Build datasets and dataloaders
# =========================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# Encode labels
# -----------------------------
ys = np.array([CLASS_TO_ID[y] for y in Ys], dtype=np.int64)
yt_train = np.array([CLASS_TO_ID[y] for y in Yt_train], dtype=np.int64)   # not used for training
yt_test  = np.array([CLASS_TO_ID[y] for y in Yt_test], dtype=np.int64)

# -----------------------------
# Domain labels
# source = 0
# target = 1
# -----------------------------
ds = np.zeros(len(Xs), dtype=np.int64)
dt_train = np.ones(len(Xt_train), dtype=np.int64)
dt_test  = np.ones(len(Xt_test), dtype=np.int64)

# -----------------------------
# Stack raw signals
# -----------------------------
Xs_np = np.stack(Xs, axis=0).astype(np.float32)
Xt_train_np = np.stack(Xt_train, axis=0).astype(np.float32)
Xt_test_np  = np.stack(Xt_test, axis=0).astype(np.float32)

print("\n================ SIGNAL SHAPES ================")
print("Source raw signals      :", Xs_np.shape)
print("Target train raw signals:", Xt_train_np.shape)
print("Target test raw signals :", Xt_test_np.shape)

# -----------------------------
# Custom dataset
# -----------------------------
class DomainDataset(Dataset):
    def __init__(self, X, y, d, Pg, Pp, use_labels=True):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.d = torch.tensor(d, dtype=torch.long)
        self.Pg = torch.tensor(Pg, dtype=torch.float32)   # (N, 16)
        self.Pp = torch.tensor(Pp, dtype=torch.float32)   # (N, 100, 4)
        self.use_labels = use_labels

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.d[idx], self.Pg[idx], self.Pp[idx]

# -----------------------------
# Create datasets
# -----------------------------
src_dataset = DomainDataset(
    X=Xs_np,
    y=ys,
    d=ds,
    Pg=Pg_s_norm,
    Pp=Pp_s,
    use_labels=True
)

tgt_train_dataset = DomainDataset(
    X=Xt_train_np,
    y=yt_train,
    d=dt_train,
    Pg=Pg_tu_norm,
    Pp=Pp_tu,
    use_labels=False
)

tgt_test_dataset = DomainDataset(
    X=Xt_test_np,
    y=yt_test,
    d=dt_test,
    Pg=Pg_tt_norm,
    Pp=Pp_tt,
    use_labels=True
)

# -----------------------------
# Dataloaders
# -----------------------------
BATCH_SIZE = 16

src_loader = DataLoader(
    src_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

tgt_train_loader = DataLoader(
    tgt_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

tgt_test_loader = DataLoader(
    tgt_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

# -----------------------------
# Sanity check one batch
# -----------------------------
xs_b, ys_b, ds_b, pgs_b, pps_b = next(iter(src_loader))
xt_b, yt_b, dt_b, pgt_b, ppt_b = next(iter(tgt_train_loader))

print("\n================ BATCH SANITY CHECK ================")
print("Source batch X :", xs_b.shape)
print("Source batch y :", ys_b.shape)
print("Source batch d :", ds_b.shape)
print("Source batch Pg:", pgs_b.shape)
print("Source batch Pp:", pps_b.shape)

print("\nTarget batch X :", xt_b.shape)
print("Target batch y :", yt_b.shape)
print("Target batch d :", dt_b.shape)
print("Target batch Pg:", pgt_b.shape)
print("Target batch Pp:", ppt_b.shape)

print("\n================ DOMAIN LABEL CHECK ================")
print("Unique source domain labels:", torch.unique(ds_b))
print("Unique target domain labels:", torch.unique(dt_b))

print("\n================ NAN CHECK ================")
print("Any NaN in source Pg:", torch.isnan(pgs_b).any().item())
print("Any NaN in source Pp:", torch.isnan(pps_b).any().item())
print("Any NaN in target Pg:", torch.isnan(pgt_b).any().item())
print("Any NaN in target Pp:", torch.isnan(ppt_b).any().item())

print("\n================ DATASET LENGTHS ================")
print("Source dataset length      :", len(src_dataset))
print("Target train dataset length:", len(tgt_train_dataset))
print("Target test dataset length :", len(tgt_test_dataset))

print("\nStep 3 completed successfully.")

Using device: cpu

================ SIGNAL SHAPES ================
Source raw signals      : (451, 2, 25600)
Target train raw signals: (356, 2, 25600)
Target test raw signals : (90, 2, 25600)

================ BATCH SANITY CHECK ================
Source batch X : torch.Size([16, 2, 25600])
Source batch y : torch.Size([16])
Source batch d : torch.Size([16])
Source batch Pg: torch.Size([16, 16])
Source batch Pp: torch.Size([16, 100, 4])

Target batch X : torch.Size([16, 2, 25600])
Target batch y : torch.Size([16])
Target batch d : torch.Size([16])
Target batch Pg: torch.Size([16, 16])
Target batch Pp: torch.Size([16, 100, 4])

================ DOMAIN LABEL CHECK ================
Unique source domain labels: tensor([0])
Unique target domain labels: tensor([1])

================ NAN CHECK ================
Any NaN in source Pg: False
Any NaN in source Pp: False
Any NaN in target Pg: False
Any NaN in target Pp: False

================ DATASET LENGTHS ================
Source dataset length    

In [4]:
# =========================================================
# STEP 4: Model definition
# - Patch embedding
# - Stable SSM backbone
# - Domain adaptive nonlinear gate
# - Fault head
# - Domain head with GRL
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# Gradient Reversal Layer
# -----------------------------
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lam * grad_output, None

def grl(x, lam=1.0):
    return GradReverse.apply(x, lam)

# -----------------------------
# Patch embedding
# input : (B, 2, 25600)
# output: (B, 100, emb_dim)
# -----------------------------
class PatchEmbed1D(nn.Module):
    def __init__(self, patch_size=256, in_ch=2, emb_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(in_ch * patch_size, emb_dim)

    def forward(self, x):
        B, C, L = x.shape
        T = L // self.patch_size
        x = x[:, :, :T * self.patch_size]
        x = x.view(B, C, T, self.patch_size).permute(0, 2, 1, 3).contiguous()
        x = x.view(B, T, C * self.patch_size)
        return self.proj(x)

# -----------------------------
# Stable SSM block
# input : (B, T, D)
# output: (B, T, D)
# -----------------------------
class StableSSM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.log_A = nn.Parameter(torch.zeros(dim))
        self.B = nn.Parameter(torch.randn(dim) * 0.1)
        self.C = nn.Parameter(torch.randn(dim) * 0.1)
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, T, D = x.shape
        h = torch.zeros(B, D, device=x.device)
        A = -torch.exp(self.log_A)

        outs = []
        for t in range(T):
            xt = x[:, t]
            g = self.gate(xt)
            h_new = torch.tanh(A * h + self.B * xt)
            h = g * h_new + (1.0 - g) * h
            yt = self.C * h
            outs.append(yt.unsqueeze(1))

        y = torch.cat(outs, dim=1)
        return self.norm(y)

# -----------------------------
# Domain adaptive nonlinear gate
# input:
#   f         : (B, emb_dim)
#   nl_global : (B, 16)
#   dom       : (B,)
# output:
#   gated f   : (B, emb_dim)
# -----------------------------
class DomainAdaptiveNonlinearGate(nn.Module):
    def __init__(self, nl_dim=16, emb_dim=64):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(nl_dim))
        self.dom_embed = nn.Embedding(2, nl_dim)

        self.mlp = nn.Sequential(
            nn.Linear(nl_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )

        self.alpha = nn.Parameter(torch.tensor(0.5, dtype=torch.float32))

    def forward(self, f, nl_global, dom):
        dom_bias = self.dom_embed(dom)
        sigma = torch.exp(self.log_sigma) + 1e-6
        nl_scaled = nl_global / sigma
        nl_cond = nl_scaled + dom_bias

        g = torch.tanh(self.mlp(nl_cond))
        f = f * (1.0 + self.alpha * g)
        return f

# -----------------------------
# Backbone
# -----------------------------
class NonlinearAdaptiveSSMBackbone(nn.Module):
    def __init__(self, emb_dim=64, nl_dim=16, patch_size=256):
        super().__init__()
        self.patch = PatchEmbed1D(
            patch_size=patch_size,
            in_ch=2,
            emb_dim=emb_dim
        )
        self.ssm1 = StableSSM(emb_dim)
        self.ssm2 = StableSSM(emb_dim)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.nl_gate = DomainAdaptiveNonlinearGate(nl_dim=nl_dim, emb_dim=emb_dim)

    def forward(self, x, nl_global, dom):
        z0 = self.patch(x)      # (B, T, D)
        z1 = self.ssm1(z0)      # (B, T, D)
        z2 = self.ssm2(z1)      # (B, T, D)

        z_patch = z2
        z_early = z1

        f = z2.transpose(1, 2)  # (B, D, T)
        f = self.pool(f).squeeze(-1)
        f = F.normalize(f, dim=1)

        f = self.nl_gate(f, nl_global, dom)
        f = F.normalize(f, dim=1)

        return f, z_patch, z_early

# -----------------------------
# Full model
# -----------------------------
class DASSMModel(nn.Module):
    def __init__(self, emb_dim=64, nl_dim=16, num_classes=4, patch_size=256):
        super().__init__()
        self.backbone = NonlinearAdaptiveSSMBackbone(
            emb_dim=emb_dim,
            nl_dim=nl_dim,
            patch_size=patch_size
        )
        self.fault_head = nn.Linear(emb_dim, num_classes)
        self.domain_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim // 2),
            nn.ReLU(),
            nn.Linear(emb_dim // 2, 2)
        )

    def forward(self, x, nl_global, dom, grl_lam=1.0):
        f, z_patch, z_early = self.backbone(x, nl_global, dom)
        fault_logits = self.fault_head(f)

        f_rev = grl(f, grl_lam)
        domain_logits = self.domain_head(f_rev)

        return fault_logits, domain_logits, f, z_patch, z_early

# -----------------------------
# Instantiate model
# -----------------------------
EMB_DIM = 64
NL_DIM = 16
NUM_CLASSES = 4
PATCH_SIZE = 256

model = DASSMModel(
    emb_dim=EMB_DIM,
    nl_dim=NL_DIM,
    num_classes=NUM_CLASSES,
    patch_size=PATCH_SIZE
).to(device)

# projection for global nonlinear alignment loss
nl_proj = nn.Sequential(
    nn.Linear(16, 64),
    nn.ReLU(),
    nn.Linear(64, 64)
).to(device)

# -----------------------------
# Sanity forward pass
# -----------------------------
xs_b, ys_b, ds_b, pgs_b, pps_b = next(iter(src_loader))
xt_b, yt_b, dt_b, pgt_b, ppt_b = next(iter(tgt_train_loader))

xs_b = xs_b.to(device)
ds_b = ds_b.to(device)
pgs_b = pgs_b.to(device)

xt_b = xt_b.to(device)
dt_b = dt_b.to(device)
pgt_b = pgt_b.to(device)

with torch.no_grad():
    flog_s, dlog_s, fs, zp_s, ze_s = model(xs_b, pgs_b, ds_b, grl_lam=1.0)
    flog_t, dlog_t, ft, zp_t, ze_t = model(xt_b, pgt_b, dt_b, grl_lam=1.0)

print("\n================ MODEL SANITY CHECK ================")
print("Source fault logits :", flog_s.shape)
print("Source domain logits:", dlog_s.shape)
print("Source feature      :", fs.shape)
print("Source patch emb    :", zp_s.shape)
print("Source early emb    :", ze_s.shape)

print("\nTarget fault logits :", flog_t.shape)
print("Target domain logits:", dlog_t.shape)
print("Target feature      :", ft.shape)
print("Target patch emb    :", zp_t.shape)
print("Target early emb    :", ze_t.shape)

print("\n================ VALUE CHECK ================")
print("Any NaN in source features:", torch.isnan(fs).any().item())
print("Any NaN in target features:", torch.isnan(ft).any().item())
print("Initial gate alpha:", float(model.backbone.nl_gate.alpha.detach().cpu()))

print("\nStep 4 completed successfully.")

Using device: cpu

================ MODEL SANITY CHECK ================
Source fault logits : torch.Size([16, 4])
Source domain logits: torch.Size([16, 2])
Source feature      : torch.Size([16, 64])
Source patch emb    : torch.Size([16, 100, 64])
Source early emb    : torch.Size([16, 100, 64])

Target fault logits : torch.Size([16, 4])
Target domain logits: torch.Size([16, 2])
Target feature      : torch.Size([16, 64])
Target patch emb    : torch.Size([16, 100, 64])
Target early emb    : torch.Size([16, 100, 64])

================ VALUE CHECK ================
Any NaN in source features: False
Any NaN in target features: False
Initial gate alpha: 0.5

Step 4 completed successfully.


In [5]:
# =========================================================
# STEP 5: Define losses
# - Fault CE
# - Domain CE
# - CORAL
# - Global nonlinear alignment
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Main cross entropy losses
# -----------------------------
ce_fault = nn.CrossEntropyLoss()
ce_domain = nn.CrossEntropyLoss()

def l2norm(x, eps=1e-8):
    return x / (torch.norm(x, dim=-1, keepdim=True) + eps)

# -----------------------------
# CORAL loss
# aligns covariance of source and target features
# -----------------------------
def coral_loss(source, target):
    """
    source: (B, D)
    target: (B, D)
    """
    src = source - source.mean(dim=0, keepdim=True)
    tgt = target - target.mean(dim=0, keepdim=True)

    cov_s = (src.T @ src) / (source.size(0) - 1 + 1e-12)
    cov_t = (tgt.T @ tgt) / (target.size(0) - 1 + 1e-12)

    d = source.size(1)
    return torch.mean((cov_s - cov_t) ** 2) / (4.0 * d * d + 1e-12)

# -----------------------------
# Global nonlinear alignment loss
# aligns learned feature with projected nonlinear descriptor
# -----------------------------
def global_nonlinear_alignment_loss(feats, nl_global, nl_proj):
    """
    feats     : (B, 64)
    nl_global : (B, 16)
    nl_proj   : maps 16 -> 64
    """
    feats_n = l2norm(feats)
    nl_emb = nl_proj(nl_global)
    nl_emb_n = l2norm(nl_emb)

    return torch.mean(1.0 - torch.sum(feats_n * nl_emb_n, dim=1))

# -----------------------------
# Sanity loss computation
# -----------------------------
model.train()
nl_proj.train()

xs_b, ys_b, ds_b, pgs_b, pps_b = next(iter(src_loader))
xt_b, yt_b, dt_b, pgt_b, ppt_b = next(iter(tgt_train_loader))

xs_b = xs_b.to(device)
ys_b = ys_b.to(device)
ds_b = ds_b.to(device)
pgs_b = pgs_b.to(device)

xt_b = xt_b.to(device)
dt_b = dt_b.to(device)
pgt_b = pgt_b.to(device)

# forward
fault_s, dom_s, fs, zp_s, ze_s = model(xs_b, pgs_b, ds_b, grl_lam=1.0)
fault_t, dom_t, ft, zp_t, ze_t = model(xt_b, pgt_b, dt_b, grl_lam=1.0)

# fault loss on source only
L_fault = ce_fault(fault_s, ys_b)

# domain loss on source + target
dom_labels = torch.cat([ds_b, dt_b], dim=0)
dom_logits = torch.cat([dom_s, dom_t], dim=0)
L_domain = ce_domain(dom_logits, dom_labels)

# coral alignment
L_coral = coral_loss(fs, ft)

# source nonlinear alignment
L_global = global_nonlinear_alignment_loss(fs, pgs_b, nl_proj)

print("================ LOSS SANITY CHECK ================")
print("Fault CE       :", float(L_fault.detach().cpu()))
print("Domain CE      :", float(L_domain.detach().cpu()))
print("CORAL          :", float(L_coral.detach().cpu()))
print("Global NL Align:", float(L_global.detach().cpu()))

print("\n================ NAN CHECK ================")
print("Fault loss NaN :", torch.isnan(L_fault).item())
print("Domain loss NaN:", torch.isnan(L_domain).item())
print("CORAL loss NaN :", torch.isnan(L_coral).item())
print("Global loss NaN:", torch.isnan(L_global).item())

print("\nStep 5 completed successfully.")

================ LOSS SANITY CHECK ================
Fault CE       : 1.3788686990737915
Domain CE      : 0.6969202160835266
CORAL          : 1.0004767014848426e-09
Global NL Align: 0.994167685508728

================ NAN CHECK ================
Fault loss NaN : False
Domain loss NaN: False
CORAL loss NaN : False
Global loss NaN: False

Step 5 completed successfully.
